In [0]:
%sql

USE CATALOG `retail-sales-proj-t9`;
USE SCHEMA gold;

In [0]:
%sql

CREATE OR REPLACE TABLE gold.sales_anomalies
USING DELTA
AS

WITH sales_stats AS (

    SELECT
        AVG(quantity) AS avg_qty,
        STDDEV(quantity) AS std_qty

    FROM gold.fact_sales

)

SELECT

    s.transaction_id,
    s.customer_id,
    s.product_id,
    s.store_id,
    s.quantity,
    s.transaction_date,

    st.avg_qty,
    st.std_qty,

    CASE

        WHEN s.quantity >
             (st.avg_qty + 2 * st.std_qty)

        THEN 'ANOMALY'

        ELSE 'NORMAL'

    END AS anomaly_status,

    CURRENT_TIMESTAMP() AS anomaly_detected_time

FROM gold.fact_sales s
CROSS JOIN sales_stats st;

In [0]:
%sql

CREATE OR REPLACE TABLE gold.anomaly_alerts
USING DELTA
AS

SELECT

    transaction_id,
    customer_id,
    product_id,
    quantity,

    'HIGH_QUANTITY_ALERT' AS alert_type,

    CURRENT_TIMESTAMP() AS alert_generated_time

FROM gold.sales_anomalies

WHERE anomaly_status = 'ANOMALY';

In [0]:
%sql

SELECT *
FROM gold.sales_anomalies
WHERE anomaly_status = 'ANOMALY';

In [0]:
%sql

CREATE OR REPLACE VIEW gold.valid_sales AS

SELECT *
FROM gold.sales_anomalies

WHERE anomaly_status = 'NORMAL';